## 1. 说明与研究目标

# Topic–country–year 面板与轨迹数据

本 notebook 从论文级表构建 **topic–country–year** 面板，并导出 `(country, topic)` 的 **rolling 平滑后**轨迹矩阵宽表，供 `country_topic_catchup_gmm.ipynb` 进行轨迹与机制 **GMM** 聚类及对照分析。

本版新增国家低参与过滤：按国家 `total_count` 的分位数阈值过滤得到主面板 `topic_country_year_panel.csv`，同时保留全量备份 `topic_country_year_panel_full.csv`。

**上游依赖**：优先使用 `output/cluster_results/paper_topics.csv`（由 `cluster_keybert_from_cluster.ipynb` 导出）。若缺失，将尝试从配置中的主 CSV 读取并检测 `topic` 列。

**产出目录**：`output/catchup_mvp/`（不覆盖 `cluster_results`）。

## 2. 导入、配置路径与输出目录

复用 `config/cluster_keybert_from_cluster.json` 中的 `paths`、`reproducibility`、`time_evolution`；本 notebook 的产出写入 `output/catchup_mvp/`。

In [30]:
from __future__ import annotations

import json
import os
import random
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Local config helpers (avoid importing scripts.cluster_pipeline package — its __init__ pulls heavy deps).


def load_json_config(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def set_reproducibility(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


_CONFIG_PATH = Path("config/cluster_keybert_from_cluster.json")
_cfg = load_json_config(_CONFIG_PATH)
_paths = _cfg["paths"]
DATA_PATH = Path(_paths["data_csv"])
CLUSTER_OUTPUT_DIR = Path(_paths["output_dir"])
PAPER_TOPICS_PATH = CLUSTER_OUTPUT_DIR / "paper_topics.csv"
TOPIC_INFO_PATH = CLUSTER_OUTPUT_DIR / "topic_info.csv"

_time = _cfg.get("time_evolution", {})
ROLLING_WINDOW = int(_time.get("rolling_window_size", 5))
ROLLING_MIN_PERIODS = int(_time.get("rolling_min_periods", 3))

RANDOM_STATE = int(_cfg["reproducibility"]["seed"])
set_reproducibility(RANDOM_STATE)

CATCHUP_MVP_DIR = Path("output/catchup_mvp")
CATCHUP_MVP_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH.resolve())
print("PAPER_TOPICS_PATH:", PAPER_TOPICS_PATH.resolve())
print("CATCHUP_MVP_DIR:", CATCHUP_MVP_DIR.resolve())
print("ROLLING_WINDOW / MIN_PERIODS:", ROLLING_WINDOW, ROLLING_MIN_PERIODS)

DATA_PATH: /Users/luoyiti/Project/catch-up/data/dataCleanSCIE.csv
PAPER_TOPICS_PATH: /Users/luoyiti/Project/catch-up/output/cluster_results/paper_topics.csv
CATCHUP_MVP_DIR: /Users/luoyiti/Project/catch-up/output/catchup_mvp
ROLLING_WINDOW / MIN_PERIODS: 5 3


## 3. 参数区（集中管理）

调节阈值时只改本节。

In [31]:
# Panel & frontier
FRONTIER_TOP_K = 2

# Sample filter for trajectory rows (country, topic) — must match country_topic_catchup_gmm.ipynb
MIN_ACTIVE_YEARS = 10  # years with count > 0

# Country-level denoising filter on panel input for downstream GMM
ENABLE_COUNTRY_FILTER = True
COUNTRY_FILTER_METRIC = "total_count"
COUNTRY_FILTER_MODE = "quantile"
COUNTRY_FILTER_QUANTILE = 0.50
COUNTRY_FILTER_OPERATOR = "ge"  # keep total_count >= threshold_value

PANEL_MAIN_NAME = "topic_country_year_panel.csv"
PANEL_FULL_BACKUP_NAME = "topic_country_year_panel_full.csv"
COUNTRY_FILTER_REPORT_NAME = "country_filter_report.csv"
COUNTRIES_DROPPED_NAME = "countries_dropped.csv"

plt.rcParams["figure.dpi"] = 120

## 4. 读取数据与字段自动识别

- 若存在 `paper_topics.csv`：直接使用。
- 否则读取主 CSV；若仍无 `topic` 列，则报错并提示先运行上游 notebook。

In [32]:
TOPIC_CANDIDATES = ("topic", "Topic", "topic_id")
YEAR_CANDIDATES = ("year", "Year", "pub_year", "publication_year")
COUNTRY_CANDIDATES = ("country_code", "country", "Country", "nation")


def _first_present(df: pd.DataFrame, names: tuple[str, ...]) -> str | None:
    for n in names:
        if n in df.columns:
            return n
    return None


def resolve_topic_country_year_columns(df: pd.DataFrame) -> dict[str, str]:
    out: dict[str, str] = {}
    out["topic"] = _first_present(df, TOPIC_CANDIDATES) or ""
    out["year"] = _first_present(df, YEAR_CANDIDATES) or ""
    out["country_src"] = _first_present(df, COUNTRY_CANDIDATES) or ""
    missing = [k for k, v in out.items() if not v]
    if missing:
        raise KeyError(f"Missing columns for: {missing}. Available: {list(df.columns)[:40]}...")
    return out  # type: ignore[return-value]


def load_paper_level_table() -> tuple[pd.DataFrame, str]:
    # Returns (df, source_label).
    if PAPER_TOPICS_PATH.exists():
        df = pd.read_csv(PAPER_TOPICS_PATH, low_memory=False)
        return df, str(PAPER_TOPICS_PATH)
    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Neither {PAPER_TOPICS_PATH} nor {DATA_PATH} exists. "
            "Run cluster_keybert_from_cluster.ipynb to export paper_topics.csv or add data CSV."
        )
    df = pd.read_csv(DATA_PATH, low_memory=False)
    if _first_present(df, TOPIC_CANDIDATES) is None:
        raise ValueError(
            f"No topic column in {DATA_PATH}. Export paper_topics.csv by running cluster_keybert_from_cluster.ipynb."
        )
    return df, str(DATA_PATH)


papers, _src = load_paper_level_table()
print("Loaded rows:", len(papers), "from", _src)
print("Columns (sample):", list(papers.columns)[:25])

cols = resolve_topic_country_year_columns(papers)
TOPIC_COL = cols["topic"]
YEAR_COL = cols["year"]
COUNTRY_SRC_COL = cols["country_src"]

# If we map country_code -> country, drop raw WoS `country` to avoid duplicate column names.
if COUNTRY_SRC_COL != "country" and "country" in papers.columns:
    papers = papers.drop(columns=["country"], errors="ignore")

papers = papers.rename(columns={TOPIC_COL: "topic", YEAR_COL: "year", COUNTRY_SRC_COL: "country"})
papers["topic"] = pd.to_numeric(papers["topic"], errors="coerce")
papers["year"] = pd.to_numeric(papers["year"], errors="coerce").astype("Int64")

before = len(papers)
papers = papers.dropna(subset=["topic", "year", "country"]).copy()
papers["topic"] = papers["topic"].astype(int)
papers["year"] = papers["year"].astype(int)
if (papers["topic"] == -1).any():
    papers = papers[papers["topic"] != -1].copy()
papers["country"] = papers["country"].astype(str).str.strip()
papers = papers[papers["country"].str.len() > 0]
print(f"Dropped {before - len(papers)} rows (missing topic/year/country or topic==-1)")

YEAR_MIN = int(papers["year"].min())
YEAR_MAX = int(papers["year"].max())

if COUNTRY_FILTER_METRIC != "total_count":
    raise ValueError(f"Unsupported COUNTRY_FILTER_METRIC: {COUNTRY_FILTER_METRIC}")
if COUNTRY_FILTER_MODE != "quantile":
    raise ValueError(f"Unsupported COUNTRY_FILTER_MODE: {COUNTRY_FILTER_MODE}")
if not (0.0 <= COUNTRY_FILTER_QUANTILE <= 1.0):
    raise ValueError("COUNTRY_FILTER_QUANTILE must be between 0 and 1")
if COUNTRY_FILTER_OPERATOR not in {"ge"}:
    raise ValueError(f"Unsupported COUNTRY_FILTER_OPERATOR: {COUNTRY_FILTER_OPERATOR}")

print("\nResolved columns: topic/year/country (renamed from)", cols)
print(f"Year span in cleaned papers: {YEAR_MIN}-{YEAR_MAX}")
print(papers.head(3))

Loaded rows: 25794 from output/cluster_results/paper_topics.csv
Columns (sample): ['paper_id', 'country', 'year', 'country_code', 'topic', 'topic_prob']
Dropped 0 rows (missing topic/year/country or topic==-1)

Resolved columns: topic/year/country (renamed from) {'topic': 'topic', 'year': 'year', 'country_src': 'country_code'}
Year span in cleaned papers: 1990-2026
              paper_id  year         country  topic  topic_prob
0  WOS:A1990EQ23400008  1990          Canada     34    0.957143
1  WOS:A1990EQ34700002  1990              US     16    0.964363
2  WOS:A1990DM80100011  1990  Czech Republic     32    0.945405


## 5. 构建 topic–country–year 面板（含国家低参与过滤）

先基于全体国家构建 **full panel**，再按国家 `total_count` 的分位数阈值进行过滤，得到 **filtered panel** 作为下游 GMM 的主输入。

- `full panel`：用于备份与可追溯。
- `filtered panel`：写入主文件 `topic_country_year_panel.csv`。

### 前沿 `frontier_value`（MVP）

对每个 `(topic, year)`：将所有国家的 `share` 从高到低排序，取 **前 `FRONTIER_TOP_K` 个国家** 的 `share` 算术平均；若有效国家数不足 `k`，则对当前所有国家取均值。

In [33]:
def frontier_topk_mean_share(shares: pd.Series, k: int = FRONTIER_TOP_K) -> float:
    # Mean of top-k country shares within one (topic, year).
    s = shares.astype(float).sort_values(ascending=False)
    s = s[s > 0] if (s > 0).any() else s
    if s.empty:
        return 0.0
    top = s.head(min(k, len(s)))
    return float(top.mean())


def build_topic_country_year_panel(
    papers: pd.DataFrame,
    year_min: int,
    year_max: int,
) -> pd.DataFrame:
    counts = (
        papers.groupby(["topic", "country", "year"], observed=True)
        .size()
        .rename("count")
        .reset_index()
    )
    topics = counts["topic"].unique()
    countries = counts["country"].unique()
    years = np.arange(year_min, year_max + 1, dtype=int)
    full = pd.MultiIndex.from_product(
        [topics, countries, years], names=["topic", "country", "year"]
    ).to_frame(index=False)
    panel = full.merge(counts, on=["topic", "country", "year"], how="left")
    panel["count"] = panel["count"].fillna(0).astype(int)

    tot = panel.groupby(["topic", "year"])["count"].transform("sum")
    panel["share"] = np.where(tot > 0, panel["count"] / tot, 0.0)
    panel["rank"] = panel.groupby(["topic", "year"])["count"].rank(ascending=False, method="min")
    panel["frontier_value"] = panel.groupby(["topic", "year"])["share"].transform(
        lambda s: frontier_topk_mean_share(s, FRONTIER_TOP_K)
    )
    panel["ratio_to_frontier"] = np.where(
        panel["frontier_value"] > 0, panel["share"] / panel["frontier_value"], np.nan
    )
    panel["gap_to_frontier"] = panel["frontier_value"] - panel["share"]
    return panel


panel_full = build_topic_country_year_panel(papers, YEAR_MIN, YEAR_MAX)
country_stats = (
    panel_full.groupby("country", as_index=False)["count"]
    .sum()
    .rename(columns={"count": "total_count"})
)
if country_stats.empty:
    raise RuntimeError("country_stats is empty. Cannot apply country filter.")

threshold_value = float(country_stats["total_count"].quantile(COUNTRY_FILTER_QUANTILE))
if COUNTRY_FILTER_OPERATOR == "ge":
    keep_mask = country_stats["total_count"] >= threshold_value
else:
    raise ValueError(f"Unsupported COUNTRY_FILTER_OPERATOR: {COUNTRY_FILTER_OPERATOR}")

if not ENABLE_COUNTRY_FILTER:
    keep_mask = pd.Series(True, index=country_stats.index)

country_stats["keep_flag"] = keep_mask.astype(bool)
country_stats["threshold_value"] = threshold_value
country_stats["filter_metric"] = COUNTRY_FILTER_METRIC
country_stats["filter_mode"] = COUNTRY_FILTER_MODE
country_stats["filter_quantile"] = COUNTRY_FILTER_QUANTILE
country_stats["filter_operator"] = COUNTRY_FILTER_OPERATOR

countries_before = int(country_stats["country"].nunique())
countries_after = int(country_stats["keep_flag"].sum())
if countries_after == 0:
    raise RuntimeError("No country retained by filter. Adjust COUNTRY_FILTER_QUANTILE.")

kept_countries = country_stats.loc[country_stats["keep_flag"], "country"].tolist()
papers_filtered = papers[papers["country"].isin(kept_countries)].copy()
panel = build_topic_country_year_panel(papers_filtered, YEAR_MIN, YEAR_MAX)

panel_main_path = CATCHUP_MVP_DIR / PANEL_MAIN_NAME
panel_full_backup_path = CATCHUP_MVP_DIR / PANEL_FULL_BACKUP_NAME
country_filter_report_path = CATCHUP_MVP_DIR / COUNTRY_FILTER_REPORT_NAME
countries_dropped_path = CATCHUP_MVP_DIR / COUNTRIES_DROPPED_NAME

panel_full.to_csv(panel_full_backup_path, index=False)
panel.to_csv(panel_main_path, index=False)

country_stats = country_stats.sort_values(["keep_flag", "total_count", "country"], ascending=[False, False, True])
country_stats.to_csv(country_filter_report_path, index=False)

countries_dropped = country_stats.loc[~country_stats["keep_flag"], ["country", "total_count"]].copy()
countries_dropped["threshold_value"] = threshold_value
countries_dropped.to_csv(countries_dropped_path, index=False)

print("Full panel shape:", panel_full.shape)
print("Filtered panel shape:", panel.shape)
print("Countries before/after:", countries_before, countries_after)
print("Country total_count threshold (P50):", threshold_value)
print("Saved:", panel_main_path.name, "(filtered main)")
print("Saved:", panel_full_backup_path.name, "(full backup)")
print("Saved:", country_filter_report_path.name)
print("Saved:", countries_dropped_path.name)

Full panel shape: (194250, 9)
Filtered panel shape: (98050, 9)
Countries before/after: 105 53
Country total_count threshold (P50): 20.0
Saved: topic_country_year_panel.csv (filtered main)
Saved: topic_country_year_panel_full.csv (full backup)
Saved: country_filter_report.csv
Saved: countries_dropped.csv


## 6. `(country, topic)` 轨迹矩阵（ratio + roll5）

以下步骤基于 **filtered panel**（即主文件 `topic_country_year_panel.csv`）构建：

- 对每个 `(country, topic)`，在完整年份轴上取 `ratio_to_frontier`；**缺失补 0**（表示该年无发文，份额为 0）。
- Rolling：对年度序列做 `rolling(ROLLING_WINDOW, min_periods=ROLLING_MIN_PERIODS).mean()`，得到 **聚类用主序列**。
- **`active_years`**：`count > 0` 的年数；仅保留 `active_years >= MIN_ACTIVE_YEARS` 进入聚类。

In [34]:
years_all = np.sort(panel["year"].unique())
year_cols = [str(y) for y in years_all]


def build_trajectory_matrix(panel: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.DataFrame]:
    # Returns wide_roll, wide_ratio_raw, active_years, count_filled (same MultiIndex as wide_roll).
    ratio_pivot = panel.pivot_table(
        index=["country", "topic"], columns="year", values="ratio_to_frontier", aggfunc="first"
    )
    ratio_pivot = ratio_pivot.reindex(columns=years_all)
    ratio_filled = ratio_pivot.fillna(0.0)

    roll = ratio_filled.T.rolling(ROLLING_WINDOW, min_periods=ROLLING_MIN_PERIODS).mean().T
    roll = roll.fillna(0.0)

    count_pivot = panel.pivot_table(
        index=["country", "topic"], columns="year", values="count", aggfunc="first"
    ).reindex(columns=years_all)
    count_filled = count_pivot.fillna(0).astype(int)
    active_years = (count_filled > 0).sum(axis=1)

    mask = active_years >= MIN_ACTIVE_YEARS
    wide_roll = roll.loc[mask].copy()
    wide_roll.columns = year_cols
    wide_ratio = ratio_filled.loc[mask].copy()
    wide_ratio.columns = year_cols
    active_years_f = active_years.loc[mask]
    count_sub = count_filled.loc[mask].copy()
    count_sub.columns = year_cols
    return wide_roll, wide_ratio, active_years_f, count_sub


wide_roll, wide_ratio, active_years_s, count_filled_sub = build_trajectory_matrix(panel)
if len(wide_roll) == 0:
    raise RuntimeError(
        "No (country, topic) pairs pass MIN_ACTIVE_YEARS; lower MIN_ACTIVE_YEARS or check paper-level data."
    )
meta = wide_roll.reset_index()[["country", "topic"]]
traj_export = pd.concat([meta.reset_index(drop=True), wide_roll.reset_index(drop=True)], axis=1)
traj_export.to_csv(CATCHUP_MVP_DIR / "country_topic_trajectory_matrix.csv", index=False)
print("Trajectory source panel:", panel_main_path.name)
print("Clustering units:", len(wide_roll), "year dim:", wide_roll.shape[1])
print("Saved country_topic_trajectory_matrix.csv")

Trajectory source panel: topic_country_year_panel.csv
Clustering units: 417 year dim: 37
Saved country_topic_trajectory_matrix.csv


## 7. 交付清单（本 notebook）

运行完成后在 `output/catchup_mvp/` 应至少存在：

| 文件 | 说明 |
|------|------|
| `topic_country_year_panel.csv` | 过滤后的 topic–country–year 主面板（供 GMM 直接读取） |
| `topic_country_year_panel_full.csv` | 全量国家面板备份 |
| `country_topic_trajectory_matrix.csv` | 基于过滤后面板、并通过 `MIN_ACTIVE_YEARS` 过滤后的 rolling 轨迹宽表 |
| `country_filter_report.csv` | 国家过滤报告（含 `total_count`、阈值与保留标记） |
| `countries_dropped.csv` | 被过滤国家清单 |

下一步：在相同参数约定下运行 `country_topic_catchup_gmm.ipynb`。

In [35]:
# Delivery summary (this notebook only)
import json

summary = {
    "notebook": "country_topic_catchup_data.ipynb",
    "inputs_tried": [str(PAPER_TOPICS_PATH), str(DATA_PATH)],
    "resolved_columns": {"topic": TOPIC_COL, "year": YEAR_COL, "country_renamed_from": COUNTRY_SRC_COL},
    "country_filter": {
        "enabled": ENABLE_COUNTRY_FILTER,
        "metric": COUNTRY_FILTER_METRIC,
        "mode": COUNTRY_FILTER_MODE,
        "quantile": COUNTRY_FILTER_QUANTILE,
        "operator": COUNTRY_FILTER_OPERATOR,
        "threshold_value": threshold_value,
        "countries_before": countries_before,
        "countries_after": countries_after,
    },
    "outputs_dir": str(CATCHUP_MVP_DIR.resolve()),
    "output_files": [
        PANEL_MAIN_NAME,
        PANEL_FULL_BACKUP_NAME,
        "country_topic_trajectory_matrix.csv",
        COUNTRY_FILTER_REPORT_NAME,
        COUNTRIES_DROPPED_NAME,
    ],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "notebook": "country_topic_catchup_data.ipynb",
  "inputs_tried": [
    "output/cluster_results/paper_topics.csv",
    "data/dataCleanSCIE.csv"
  ],
  "resolved_columns": {
    "topic": "topic",
    "year": "year",
    "country_renamed_from": "country_code"
  },
  "country_filter": {
    "enabled": true,
    "metric": "total_count",
    "mode": "quantile",
    "quantile": 0.5,
    "operator": "ge",
    "threshold_value": 20.0,
    "countries_before": 105,
    "countries_after": 53
  },
  "outputs_dir": "/Users/luoyiti/Project/catch-up/output/catchup_mvp",
  "output_files": [
    "topic_country_year_panel.csv",
    "topic_country_year_panel_full.csv",
    "country_topic_trajectory_matrix.csv",
    "country_filter_report.csv",
    "countries_dropped.csv"
  ]
}
